### [Dependencies]

In [1]:
import os
import umap
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from tqdm.notebook import tqdm
from scipy.signal import savgol_filter
from sklearn.decomposition import PCA

bee = 'Jataí' #'Mandaçaia' #'Ambas'

os.makedirs('../graficos/',          exist_ok=True)
os.makedirs(f'../graficos/{bee}', exist_ok=True)

load_path = '../..'
save_path = f'../graficos/{bee}'


warnings.filterwarnings('ignore', message='.*n_jobs value 1 overridden.*')

### [Read data]

In [2]:
if bee == 'Ambas':
    df = []
    for aux_bee in ['Jataí', 'Mandaçaia']:
            
        aux_df = pd.read_excel(f'{load_path}/{aux_bee}_infravermelho_duas_coletas.xlsx')
        aux_df = aux_df.T.reset_index()
        
        columns = aux_df.iloc[0].tolist()
        columns[0:3] = ['Abelha', 'Dia_coleta', 'IDX_produtor'] if aux_bee == 'Mandaçaia' else ['Dia_coleta', 'Abelha', 'IDX_produtor']
        
        aux_df.columns = columns
        aux_df = aux_df[1:]
        
        aux_df['Rodada_coleta'] = aux_df['Dia_coleta'].str.extract(r'(\d+)º').astype(int).values
        aux_df['IDX_produtor']  = aux_df['IDX_produtor'].str.replace('Produtor ', '', regex=False).astype(int)
        aux_df['Dia_coleta']    = aux_df.groupby('Rodada_coleta').cumcount() + 1

        df.append(aux_df)

    df = pd.concat(df).dropna(axis=1)
    df['Abelha'] = ['Jataí' for _ in range(16)] + ['Mandaçaia' for _ in range(16)]
    
else: 
    df = pd.read_excel(f'{load_path}/{bee}_infravermelho_duas_coletas.xlsx')
    df = df.T.reset_index()
    
    columns = df.iloc[0].tolist()
    columns[0:3] = ['Abelha', 'Dia_coleta', 'IDX_produtor'] if bee == 'Mandaçaia' else ['Dia_coleta', 'Abelha', 'IDX_produtor']
    
    df.columns = columns
    df = df[1:]
    
    df['Rodada_coleta'] = df['Dia_coleta'].str.extract(r'(\d+)º').astype(int).values
    df['IDX_produtor'] = df['IDX_produtor'].str.replace('Produtor ', '', regex=False).astype(int)
    df['Dia_coleta'] = df.groupby('Rodada_coleta').cumcount() + 1

    if bee == 'Mandaçaia':
        df['Abelha'] = ['Mandaçaia' for _ in range(16)]
    
    df.head()

### [Signal Visualization]

In [3]:
metadata_cols = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
freq_cols     = [col for col in df.columns if col not in metadata_cols]

df_plot = df.melt(
    id_vars=metadata_cols, 
    value_vars=freq_cols, 
    var_name='Frequencia_cm1', 
    value_name='Intensidade'
)

df_plot['Frequencia_cm1'] = df_plot['Frequencia_cm1'].astype(float)

df_plot['Legenda'] = 'Produtor ' + df_plot['IDX_produtor'].astype(str)

df_plot['Rodada_coleta'] = df_plot['Rodada_coleta'].map({1: '1º Dia', 2: '2º Dia'})

fig = px.line(
    df_plot,
    x='Frequencia_cm1',
    y='Intensidade',
    color='Legenda',        
    line_dash='Rodada_coleta', 
    hover_name='Abelha',
    title=f'Espectro de Frequência das Amostras de Mel da Abelha {bee} (FTIR)',
    labels={
        'Frequencia_cm1': 'Número de Onda (cm⁻¹)',
        'Intensidade': 'Intensidade / Absorbância',
        'Rodada_coleta': 'Coleta' 
    },
    template='plotly_white'
)

fig.update_layout(
    xaxis=dict(autorange='reversed'),
    legend_title_text='Amostras',
    hovermode="x unified" 
)

fig.update_traces(line=dict(width=1.5))

fig.update_layout(
    width=1000,  
    height=700, 
)
fig.write_html(f"{save_path}/Frequencias.html")

### [PCA Visualization]

#### [2D]

In [4]:
freqs = [
    (3800, 3015), (3015, 2450), (1770, 1530), 
    (1520, 1200), (1200, 905), (905, 700), 
    ('Todas', 'Todas')
]

for freq_max, freq_min in tqdm(freqs, total=len(freqs)):
    cols_metadados = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
    all_freqs      = [col for col in df.columns if col not in cols_metadados]

    if freq_max == 'Todas':
        freq_selected = all_freqs
        file_name = "Todas"
    else:
        freq_selected = [
            col for col in all_freqs 
            if freq_min <= float(col) <= freq_max
        ]
        file_name = f"{freq_max}_{freq_min}"

    filtered = df[freq_selected].values

    redutor = PCA(n_components=2, random_state=42)
    
    df_2D = pd.DataFrame(
        data=redutor.fit_transform(filtered), 
        columns=['Component 1', 'Component 2']
    )
   
    df_2D['IDX_produtor']  = ('Produtor ' + df['IDX_produtor'].astype(str)).values
    df_2D['Rodada_coleta'] = (df['Rodada_coleta'].astype(str) + 'ª Coleta').values
    df_2D['Abelha']        = df['Abelha'].values

    fig_prod = px.scatter(
        df_2D, x='Component 1', y='Component 2',
        color='IDX_produtor', text='Rodada_coleta', hover_name='Abelha',
        hover_data={'Component 1': False, 'Component 2': False}
    )

    fig_rodada = px.scatter(
        df_2D, x='Component 1', y='Component 2',
        color='Rodada_coleta', text='IDX_produtor', hover_name='Abelha',
        hover_data={'Component 1': False, 'Component 2': False}
    )

    if bee == 'Ambas':
        fig_abelha = px.scatter(
            df_2D, x='Component 1', y='Component 2',
            color='Abelha', text='Rodada_coleta', hover_name='IDX_produtor',
            hover_data={'Component 1': False, 'Component 2': False}
        )
    

    fig = go.Figure()

    for trace in fig_prod.data:
        fig.add_trace(trace)

    for trace in fig_rodada.data:
        trace.visible = False
        fig.add_trace(trace)

    if bee == 'Ambas':
        for trace in fig_abelha.data:
            trace.visible = False
            fig.add_trace(trace)

    num_prod_traces   = len(fig_prod.data)
    num_rodada_traces = len(fig_rodada.data)
    num_abelha_traces = len(fig_abelha.data) if bee == 'Ambas' else 0
    

    show_produtor = [True]  * num_prod_traces + [False] * num_rodada_traces + [False] * num_abelha_traces
    show_rodada   = [False] * num_prod_traces + [True]  * num_rodada_traces + [False] * num_abelha_traces

    if bee == 'Ambas': 
        show_abelha = [False] * num_prod_traces + [False] * num_rodada_traces + [True] * num_abelha_traces


    my_buttons = [
        dict(
            label="Agrupar por Produtor",
            method="update",
            args=[{"visible": show_produtor}]
        ),
        dict(
            label="Agrupar por Coleta",
            method="update",
            args=[{"visible": show_rodada}]
        )
    ]


    if bee == 'Ambas':
        my_buttons.append(
            dict(
                label="Agrupar por Abelha",
                method="update",
                args=[{"visible": show_abelha}]
            )
        )


    fig.update_layout(
        title=f'Análise PCA da {bee}: Agrupamento de Produtores e Coletas',
        template='plotly_white',
        width=700, 
        height=700,
        xaxis_title=f'Componente Principal 1 ({redutor.explained_variance_ratio_[0]*100:.1f}%)',
        yaxis_title=f'Componente Principal 2 ({redutor.explained_variance_ratio_[1]*100:.1f}%)',
        yaxis=dict(scaleanchor="x", scaleratio=1),
        
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.5,
                y=-0.15,
                xanchor="center",
                yanchor="top",
                showactive=True,
                buttons=my_buttons 
            )
        ]
    )

    fig.update_traces(
        textposition='top center',
        textfont_size=9, 
        marker=dict(size=10, opacity=0.8, line=dict(width=1, color='DarkSlateGrey'))
    )

    fig.write_html(f"{save_path}/PCA_2D_{file_name}.html")

  0%|          | 0/7 [00:00<?, ?it/s]

#### [3D]

In [5]:
freqs = [
    (3800, 3015), (3015, 2450), (1770, 1530), 
    (1520, 1200), (1200, 905), (905, 700), 
    ('Todas', 'Todas')
]

for freq_max, freq_min in tqdm(freqs, total=len(freqs)):
    cols_metadados = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
    all_freqs      = [col for col in df.columns if col not in cols_metadados]

    if freq_max == 'Todas':
        freq_selected = all_freqs
        file_name = "Todas"
    else:
        freq_selected = [
            col for col in all_freqs 
            if freq_min <= float(col) <= freq_max
        ]
        file_name = f"{freq_max}_{freq_min}"

    filtered = df[freq_selected].values

    redutor = PCA(n_components=3, random_state=42)
    
    df_3D = pd.DataFrame(
        data=redutor.fit_transform(filtered), 
        columns=['Component 1', 'Component 2', 'Component 3']
    )
   
    df_3D['IDX_produtor']  = ('Produtor ' + df['IDX_produtor'].astype(str)).values
    df_3D['Rodada_coleta'] = (df['Rodada_coleta'].astype(str) + 'ª Coleta').values
    df_3D['Abelha']        = df['Abelha'].values

    fig_prod = px.scatter_3d(
        df_3D, 
        x='Component 1', y='Component 2', z='Component 3',
        color='IDX_produtor', text='Rodada_coleta', hover_name='Abelha',
        hover_data={'Component 1': False, 'Component 2': False, 'Component 3': False}
    )

    fig_rodada = px.scatter_3d(
        df_3D, 
        x='Component 1', y='Component 2', z='Component 3',
        color='Rodada_coleta', text='IDX_produtor', hover_name='Abelha',
        hover_data={'Component 1': False, 'Component 2': False, 'Component 3': False}
    )

    if bee == 'Ambas':
        fig_abelha = px.scatter_3d(
            df_3D, 
            x='Component 1', y='Component 2', z='Component 3',
            color='Abelha', text='Rodada_coleta', hover_name='IDX_produtor',
            hover_data={'Component 1': False, 'Component 2': False, 'Component 3': False}
        )

    fig = go.Figure()

    for trace in fig_prod.data:
        fig.add_trace(trace)

    for trace in fig_rodada.data:
        trace.visible = False
        fig.add_trace(trace)

    if bee == 'Ambas':
        for trace in fig_abelha.data:
            trace.visible = False
            fig.add_trace(trace)

    num_prod_traces   = len(fig_prod.data)
    num_rodada_traces = len(fig_rodada.data)
    num_abelha_traces = len(fig_abelha.data) if bee == 'Ambas' else 0

    show_produtor = [True]  * num_prod_traces + [False] * num_rodada_traces + [False] * num_abelha_traces
    show_rodada   = [False] * num_prod_traces + [True]  * num_rodada_traces + [False] * num_abelha_traces

    if bee == 'Ambas': 
        show_abelha = [False] * num_prod_traces + [False] * num_rodada_traces + [True] * num_abelha_traces

    my_buttons = [
        dict(
            label="Agrupar por Produtor",
            method="update",
            args=[{"visible": show_produtor}]
        ),
        dict(
            label="Agrupar por Coleta",
            method="update",
            args=[{"visible": show_rodada}]
        )
    ]

    if bee == 'Ambas':
        my_buttons.append(
            dict(
                label="Agrupar por Abelha",
                method="update",
                args=[{"visible": show_abelha}]
            )
        )

    fig.update_layout(
        title=f'Análise PCA da {bee}: Agrupamento de Produtores e Coletas',
        template='plotly_white',
        width=800, 
        height=800,
        scene=dict(
            xaxis_title=f'Componente Principal 1 ({redutor.explained_variance_ratio_[0]*100:.1f}%)',
            yaxis_title=f'Componente Principal 2 ({redutor.explained_variance_ratio_[1]*100:.1f}%)',
            zaxis_title=f'Componente Principal 3 ({redutor.explained_variance_ratio_[2]*100:.1f}%)',
            aspectmode='cube'
        ),
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.5,
                y=-0.15,
                xanchor="center",
                yanchor="top",
                showactive=True,
                buttons=my_buttons
            )
        ]
    )

    fig.update_traces(
        textposition='top center',
        textfont_size=9, 
        marker=dict(size=6, opacity=0.8, line=dict(width=1, color='DarkSlateGrey'))
    )

    fig.write_html(f"{save_path}/PCA_3D_{file_name}.html")

  0%|          | 0/7 [00:00<?, ?it/s]

### [UMAP Visualization]

#### [2D]

In [6]:
freqs = [
    (3800, 3015), (3015, 2450), (1770, 1530), 
    (1520, 1200), (1200, 905), (905, 700), 
    ('Todas', 'Todas')
]

vizinhos_list = [3, 4, 5] if bee != 'Ambas' else [5, 7, 10, 15]
metricas_list = ['cosine', 'euclidean']
min_dist_list = [0.1, 0.8]

for freq_max, freq_min in tqdm(freqs, total=len(freqs)):
    cols_metadados = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
    all_freqs      = [col for col in df.columns if col not in cols_metadados]

    if freq_max == 'Todas':
        freq_selected = all_freqs
        file_name = "Todas"
    else:
        freq_selected = [
            col for col in all_freqs 
            if freq_min <= float(col) <= freq_max
        ]
        file_name = f"{freq_max}_{freq_min}"

    filtered = df[freq_selected].values

    for nn in vizinhos_list:
        for metrica in metricas_list:
            for md in min_dist_list:
                            
                redutor = umap.UMAP(
                                n_neighbors=nn,
                                n_components=2, 
                                metric=metrica,
                                min_dist=md,
                                random_state=42)
                
                df_2D = pd.DataFrame(
                    data=redutor.fit_transform(filtered), 
                    columns=['Component 1', 'Component 2']
                )
               
                df_2D['IDX_produtor']  = ('Produtor ' + df['IDX_produtor'].astype(str)).values
                df_2D['Rodada_coleta'] = (df['Rodada_coleta'].astype(str) + 'ª Coleta').values
                df_2D['Abelha']        = df['Abelha'].values
            
                fig_prod = px.scatter(
                    df_2D, x='Component 1', y='Component 2',
                    color='IDX_produtor', text='Rodada_coleta', hover_name='Abelha',
                    hover_data={'Component 1': False, 'Component 2': False}
                )
            
                fig_rodada = px.scatter(
                    df_2D, x='Component 1', y='Component 2',
                    color='Rodada_coleta', text='IDX_produtor', hover_name='Abelha',
                    hover_data={'Component 1': False, 'Component 2': False}
                )

                if bee == 'Ambas':
                    fig_abelha = px.scatter(
                        df_2D, x='Component 1', y='Component 2',
                        color='Abelha', text='Rodada_coleta', hover_name='IDX_produtor',
                        hover_data={'Component 1': False, 'Component 2': False}
                    )
        
                fig = go.Figure()
            
                for trace in fig_prod.data:
                    fig.add_trace(trace)
            
                for trace in fig_rodada.data:
                    trace.visible = False
                    fig.add_trace(trace)
                
                if bee == 'Ambas':
                    for trace in fig_abelha.data:
                        trace.visible = False
                        fig.add_trace(trace)
            
                num_prod_traces   = len(fig_prod.data)
                num_rodada_traces = len(fig_rodada.data)
                num_abelha_traces = len(fig_abelha.data) if bee == 'Ambas' else 0
            
                show_produtor = [True]  * num_prod_traces + [False] * num_rodada_traces
                show_rodada   = [False] * num_prod_traces + [True]  * num_rodada_traces
                
                if bee == 'Ambas': 
                    show_abelha = [False] * num_prod_traces + [False] * num_rodada_traces + [True] * num_abelha_traces

                
                my_buttons = [
                    dict(
                        label="Agrupar por Produtor",
                        method="update",
                        args=[{"visible": show_produtor}]
                    ),
                    dict(
                        label="Agrupar por Coleta",
                        method="update",
                        args=[{"visible": show_rodada}]
                    )
                ]
            
            
                if bee == 'Ambas':
                    my_buttons.append(
                        dict(
                            label="Agrupar por Abelha",
                            method="update",
                            args=[{"visible": show_abelha}]
                        )
                    )
        
                fig.update_layout(
                    title=f'Análise UMAP da {bee}: Agrupamento de Produtores e Coletas',
                    template='plotly_white',
                    width=700, 
                    height=700,
                    xaxis_title=f'Componente Principal 1',
                    yaxis_title=f'Componente Principal 2',
                    yaxis=dict(scaleanchor="x", scaleratio=1),
                    
                    updatemenus=[
                        dict(
                            type="buttons",
                            direction="right",
                            x=0.5,
                            y=-0.15,
                            xanchor="center",
                            yanchor="top",
                            showactive=True,
                            buttons=my_buttons 
                        )
                    ]
                )
            
                fig.update_traces(
                    textposition='top center',
                    textfont_size=9, 
                    marker=dict(size=10, opacity=0.8, line=dict(width=1, color='DarkSlateGrey'))
                )
            
                fig.write_html(f"{save_path}/UMAP_2D_{file_name}_{metrica}_nn{nn}_md{md}.html")

  0%|          | 0/7 [00:00<?, ?it/s]

#### [3D]

In [7]:
freqs = [
    (3800, 3015), (3015, 2450), (1770, 1530), 
    (1520, 1200), (1200, 905), (905, 700), 
    ('Todas', 'Todas')
]

vizinhos_list = [3, 4, 5] if bee != 'Ambas' else [5, 7, 10, 15]
metricas_list = ['cosine', 'euclidean']
min_dist_list = [0.1, 0.8]

for freq_max, freq_min in tqdm(freqs, total=len(freqs)):
    cols_metadados = ['Dia_coleta', 'Abelha', 'IDX_produtor', 'Rodada_coleta']
    all_freqs      = [col for col in df.columns if col not in cols_metadados]

    if freq_max == 'Todas':
        freq_selected = all_freqs
        file_name = "Todas"
    else:
        freq_selected = [
            col for col in all_freqs 
            if freq_min <= float(col) <= freq_max
        ]
        file_name = f"{freq_max}_{freq_min}"

    filtered = df[freq_selected].values

    for nn in vizinhos_list:
        for metrica in metricas_list:
            for md in min_dist_list:
                            
                redutor = umap.UMAP(
                                n_neighbors=nn,
                                n_components=3, 
                                metric=metrica,
                                min_dist=md,
                                random_state=42)
                
                df_3D = pd.DataFrame(
                    data=redutor.fit_transform(filtered), 
                    columns=['Component 1', 'Component 2', 'Component 3']
                )
               
                df_3D['IDX_produtor']  = ('Produtor ' + df['IDX_produtor'].astype(str)).values
                df_3D['Rodada_coleta'] = (df['Rodada_coleta'].astype(str) + 'ª Coleta').values
                df_3D['Abelha']        = df['Abelha'].values
            
                fig_prod = px.scatter_3d(
                    df_3D, x='Component 1', y='Component 2', z='Component 3',
                    color='IDX_produtor', text='Rodada_coleta', hover_name='Abelha',
                    hover_data={'Component 1': False, 'Component 2': False, 'Component 3': False}
                )
            
                fig_rodada = px.scatter_3d(
                    df_3D, x='Component 1', y='Component 2', z='Component 3',
                    color='Rodada_coleta', text='IDX_produtor', hover_name='Abelha',
                    hover_data={'Component 1': False, 'Component 2': False, 'Component 3': False}
                )

                if bee == 'Ambas':
                    fig_abelha = px.scatter_3d(
                        df_3D, 
                        x='Component 1', y='Component 2', z='Component 3',
                        color='Abelha', text='Rodada_coleta', hover_name='IDX_produtor',
                        hover_data={'Component 1': False, 'Component 2': False, 'Component 3': False}
                    )
                    
                fig = go.Figure()
            
                for trace in fig_prod.data:
                    fig.add_trace(trace)
            
                for trace in fig_rodada.data:
                    trace.visible = False
                    fig.add_trace(trace)

                if bee == 'Ambas':
                    for trace in fig_abelha.data:
                        trace.visible = False
                        fig.add_trace(trace)
            
                num_prod_traces   = len(fig_prod.data)
                num_rodada_traces = len(fig_rodada.data)
                num_abelha_traces = len(fig_abelha.data) if bee == 'Ambas' else 0
                
                show_produtor = [True]  * num_prod_traces + [False] * num_rodada_traces
                show_rodada   = [False] * num_prod_traces + [True]  * num_rodada_traces
                
                if bee == 'Ambas': 
                    show_abelha = [False] * num_prod_traces + [False] * num_rodada_traces + [True] * num_abelha_traces

                my_buttons = [
                    dict(
                        label="Agrupar por Produtor",
                        method="update",
                        args=[{"visible": show_produtor}]
                    ),
                    dict(
                        label="Agrupar por Coleta",
                        method="update",
                        args=[{"visible": show_rodada}]
                    )
                ]
            
                if bee == 'Ambas':
                    my_buttons.append(
                        dict(
                            label="Agrupar por Abelha",
                            method="update",
                            args=[{"visible": show_abelha}]
                        )
                    )
        
                fig.update_layout(
                    title=f'Análise UMAP da {bee}: Agrupamento de Produtores e Coletas',
                    template='plotly_white',
                    width=800, 
                    height=800,
                    scene=dict(
                        xaxis_title=f'Componente Principal 1',
                        yaxis_title=f'Componente Principal 2',
                        zaxis_title=f'Componente Principal 3',
                        aspectmode='cube'
                    ),
                    updatemenus=[
                        dict(
                            type="buttons",
                            direction="right",
                            x=0.5,
                            y=-0.15,
                            xanchor="center",
                            yanchor="top",
                            showactive=True,
                            buttons=my_buttons
                        )
                    ]
                )
            
                fig.update_traces(
                    textposition='top center',
                    textfont_size=9, 
                    marker=dict(size=10, opacity=0.8, line=dict(width=1, color='DarkSlateGrey'))
                )
            
                fig.write_html(f"{save_path}/UMAP_3D_{file_name}_{metrica}_nn{nn}_md{md}.html")

  0%|          | 0/7 [00:00<?, ?it/s]